# Bvarta Bahari — Ekspansi Rute Baru (New Route Expansion)
**Case Study: Evaluasi & Optimasi Rute Laut**  
*Oleh: Fadilla Zundina*

## 1. Pendahuluan & Latar Belakang
Proyek "Bvarta Bahari" kini memasuki tahapan strategis: **Ekspansi Jaringan**. Setelah kita mengoptimasi rute yang sudah ada, direksi membutuhkan rekomendasi 3 (tiga) rute pelayaran laut baru yang paling potensial secara finansial dan operasional.

Pada notebook ini, kita akan:
1. Mengidentifikasi seluruh kombinasi pelabuhan yang belum terhubung.
2. Menyaring rute potensial menggunakan **Model Gravitasi & Mobilitas Spasial**.
3. Melakukan prakiraan permintaan (Forecast Demand) menggunakan model regresi yang dilatih dari data rute eksisting.
4. Melakukan simulasi *Return on Investment* (ROI) dengan mencocokkan kapal yang paling efisien dari armada kita yang memenuhi batasan draft fisik pelabuhan baru tersebut.

In [1]:
import pandas as pd
import numpy as np
import os
import math
from math import radians, cos, sin, asin, sqrt
from sklearn.linear_model import LinearRegression

import warnings
warnings.filterwarnings('ignore')

data_dir = "data"
output_dir = "output"
os.makedirs(output_dir, exist_ok=True)

ports = pd.read_csv(os.path.join(data_dir, "ports.csv"))
routes = pd.read_csv(os.path.join(data_dir, "routes_existing.csv"))
mobility = pd.read_csv(os.path.join(data_dir, "mobility_daily.csv"))
fleet = pd.read_csv(os.path.join(data_dir, "fleet.csv"))
prices = pd.read_csv(os.path.join(data_dir, "route_prices.csv"))
opex = pd.read_csv(os.path.join(data_dir, "route_opex_monthly.csv"))
orders = pd.read_csv(os.path.join(data_dir, "orders_history_daily.csv"))
tides = pd.read_csv(os.path.join(data_dir, "tides_hourly.csv"))

print("Data berhasil dimuat.")

Data berhasil dimuat.


## 2. Pemodelan Spasial: Jarak Laut & Potensi Mobilitas

### Logika Teoretis Model Mobilitas/Gravitasi
Untuk mengestimasi permintaan rute yang belum pernah beroperasi, kita tidak bisa mengandalkan data penjualan tiket historis (karena datanya nol). Sebagai gantinya, kita menggunakan pendekatan spasial:
- **Mobilitas Antar Wilayah (`mobility_daily.csv`)**: Data pergerakan masyarakat harian antar region (provinsi/area) adalah indikator *leading* terkuat untuk potensi penumpang kapal laut. Rute yang menghubungkan dua pelabuhan di region dengan pergerakan masyarakat tinggi akan memiliki _latent demand_ yang besar.
- **Model Gravitasi Sederhana**: Jika data mobilitas antar region tidak tersedia, kita dapat menggunakan rasio kelas pelabuhan (`Utama`, `Pengumpul`, `Pengumpan`) berbanding terbalik dengan kuadrat jaraknya.

### Justifikasi "Faktor Koreksi Jarak"
Secara matematis, jarak terpendek antara dua titik koordinat di bumi dihitung dengan rumus **Haversine** (jarak garis lurus/udara). Namun, kapal laut tidak berlayar membelah daratan. Kapal harus mengikuti kontur pesisir pulau, memutar tanjung, dan menghindari karang. 

Pada analisis EDA pertama (`01_exploratory_data_analysis.ipynb`), kita telah melakukan regresi linier antara jarak Haversine vs jarak pelayaran laut aktual di rute eksisting, dan menemukan **Faktor Koreksi Jarak = 1.2744**. 
Oleh karena itu, setiap estimasi jarak Haversine untuk rute baru **WAJIB dikalikan dengan 1.2744** untuk mendapatkan estimasi *Nautical Miles (nm)* pelayaran laut riil yang masuk akal dan mencegah perhitungan OPEX yang terlalu rendah (under-budgeting).

In [2]:
# Fungsi Haversine
def haversine_nm(lon1, lat1, lon2, lat2):
    lon1, lat1, lon2, lat2 = map(radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1 
    dlat = lat2 - lat1 
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * asin(sqrt(a)) 
    r = 6371 # Radius bumi dalam km
    dist_km = c * r
    return dist_km * 0.539957 # konversi km ke nm

# 1. Identifikasi Rute Eksisting
existing_pairs = set()
for _, row in routes.iterrows():
    p1, p2 = sorted([row['origin_port_id'], row['dest_port_id']])
    existing_pairs.add((p1, p2))

# 2. Hasilkan Semua Kombinasi Pelabuhan (Unconnected)
all_ports = ports.to_dict('records')
unconnected = []
class_map = {'Utama': 3, 'Pengumpul': 2, 'Pengumpan': 1}

for i in range(len(all_ports)):
    for j in range(i+1, len(all_ports)):
        orig = all_ports[i]
        dest = all_ports[j]
        
        p1_id, p2_id = sorted([orig['port_id'], dest['port_id']])
        if (p1_id, p2_id) not in existing_pairs:
            # Jarak Lurus (Haversine)
            dist_straight = haversine_nm(orig['lon'], orig['lat'], dest['lon'], dest['lat'])
            
            # KOREKSI JARAK LAUT RIIL (Faktor 1.2744)
            dist_corrected = dist_straight * 1.2744
            
            # Kita singkirkan rute yang terlalu jauh secara logis (> 1000 nm) atau terlalu dekat (< 10 nm)
            if dist_corrected > 1000 or dist_corrected < 10:
                continue
                
            # Dapatkan Mobilitas Harian Rata-rata antar Region
            mob = mobility[(
                ((mobility['origin_region'] == orig['region']) & (mobility['dest_region'] == dest['region'])) |
                ((mobility['origin_region'] == dest['region']) & (mobility['dest_region'] == orig['region']))
            )]
            avg_mob = mob['estimated_travelers'].mean() if len(mob) > 0 else 0
            
            # Model Gravitasi Fallback
            gravity = (class_map.get(orig['port_class'], 1) * class_map.get(dest['port_class'], 1)) / max(1, dist_corrected)
            
            # Menentukan Max Draft Minimum Rute
            unconnected.append({
                'origin_port_id': orig['port_id'],
                'dest_port_id': dest['port_id'],
                'origin_region': orig['region'],
                'dest_region': dest['region'],
                'distance_nm': dist_corrected,
                'avg_daily_mobility': avg_mob,
                'gravity_score': gravity
            })

unconn_df = pd.DataFrame(unconnected)

# 3. Pilih Top 3 Rute Baru Berdasarkan Mobilitas (dan Gravitasi jika seri)
top_3_routes = unconn_df.sort_values(by=['avg_daily_mobility', 'gravity_score'], ascending=[False, False]).head(3).copy()
top_3_routes = top_3_routes.reset_index(drop=True)
top_3_routes['new_route_id'] = ['N01', 'N02', 'N03']

print("Top 3 Kandidat Rute Baru Terpilih:")
display(top_3_routes[['new_route_id', 'origin_port_id', 'dest_port_id', 'distance_nm', 'avg_daily_mobility']])

Top 3 Kandidat Rute Baru Terpilih:


,new_route_id,origin_port_id,dest_port_id,distance_nm,avg_daily_mobility
0,N01,PRK,MRK,67.737349,2168.178832
1,N02,PRK,BKH,87.685866,1943.737226
2,N03,PRK,TGL,181.269226,1552.655109


## 3. Analisis Kelayakan Finansial (Forecast Revenue vs Estimasi OPEX)

Untuk mengusulkan sebuah rute ke ranah bisnis, mobilitas saja tidak cukup. Rute harus menghasilkan *Return on Investment* (ROI) yang positif.

### Regresi Forecast Permintaan (Demand) & Harga
Karena kita tidak punya mesin waktu, kita akan melatih model **Regresi Spasial (Linear Regression)** menggunakan data dari rute eksisting. Model ini memetakan variabel `[Mobilitas_Harian, Jarak_NM]` menjadi target `[Permintaan_Tiket_Mingguan]`. Setelah model mengenali pola konversi mobilitas menjadi tiket dari pasar saat ini, kita aplikasikan model ini ke 3 kandidat rute baru untuk mendapatkan estimasi *Weekly Demand*.

Harga tiket rute baru diestimasi dengan mengalikan *Jarak Laut Riil (NM)* dengan *Rata-rata Harga Tiket per NM* dari seluruh rute eksisting Bvarta Bahari.

### Pemilihan Armada & Trade-off OPEX
Tantangan teknisnya adalah mencocokkan kapal dari armada eksisting:
- **Kesesuaian Draft Fisik**: Kedalaman air pelabuhan (termasuk surut ekstrem) adalah batasan kaku. Kapal yang *draft*-nya lebih dalam dari dasar pelabuhan akan kandas.
- **Efisiensi OPEX**: Kita akan mengevaluasi setiap kapal yang memenuhi syarat draft, lalu menghitung estimasi OPEX-nya berdasarkan konsumsi bahan bakar historis per mil (LPH disesuaikan ke mil laut via kecepatan kapal). Kapal yang memberikan OPEX terendah per trip namun sanggup menampung *forecast demand* akan dipilih.

ROI dihitung dengan formula: `(Estimasi Revenue Bulanan - Estimasi OPEX Bulanan) / Estimasi OPEX Bulanan * 100%`.

In [3]:
# 1. Bangun Dataset Rute Eksisting untuk Latih Regresi Demand & Harga
orders['date'] = pd.to_datetime(orders['date'])
weeks_total = (orders['date'].max() - orders['date'].min()).days / 7
weekly_demand = orders.groupby('route_id')['tickets_sold'].sum().reset_index()
weekly_demand['avg_weekly_demand'] = weekly_demand['tickets_sold'] / weeks_total

routes_model = routes.merge(weekly_demand, on='route_id')

ports_info = ports[['port_id', 'region']]
routes_model = routes_model.merge(ports_info, left_on='origin_port_id', right_on='port_id').rename(columns={'region': 'origin_region'}).drop(columns=['port_id'])
routes_model = routes_model.merge(ports_info, left_on='dest_port_id', right_on='port_id').rename(columns={'region': 'dest_region'}).drop(columns=['port_id'])

# Hitung mobilitas eksisting
mob_dict = {}
for idx, r in mobility.iterrows():
    key = tuple(sorted([r['origin_region'], r['dest_region']]))
    if key not in mob_dict:
        mob_dict[key] = []
    mob_dict[key].append(r['estimated_travelers'])
avg_mob_dict = {k: np.mean(v) for k, v in mob_dict.items()}

routes_model['avg_daily_mobility'] = routes_model.apply(lambda r: avg_mob_dict.get(tuple(sorted([r['origin_region'], r['dest_region']])), 0), axis=1)

# Harga tiket tertimbang
weighted_prices = []
for r_id in routes_model['route_id'].unique():
    r_prices = prices[prices['route_id'] == r_id]
    w_price = 0.8 * r_prices[r_prices['ticket_class'] == 'Ekonomi']['price_idr'].values[0] + 0.2 * r_prices[r_prices['ticket_class'] == 'Bisnis']['price_idr'].values[0]
    weighted_prices.append({'route_id': r_id, 'avg_ticket_price': w_price})
routes_model = routes_model.merge(pd.DataFrame(weighted_prices), on='route_id')
routes_model['price_per_nm'] = routes_model['avg_ticket_price'] / routes_model['distance_nm']
avg_price_per_nm = routes_model['price_per_nm'].mean()

# Latih Model Regresi Demand
X_train = routes_model[['avg_daily_mobility', 'distance_nm']]
y_train = routes_model['avg_weekly_demand']
demand_model = LinearRegression()
demand_model.fit(X_train, y_train)

# Persiapkan data draft dari ports
min_tides = tides.groupby('port_id')['water_level_m'].min().reset_index()
ports_tides = ports.merge(min_tides, on='port_id', how='left')
ports_tides['effective_max_draft'] = ports_tides.apply(
    lambda row: row['max_draft_m'] + row['water_level_m'] if pd.notnull(row['water_level_m']) and row['water_level_m'] < 0 else row['max_draft_m'],
    axis=1
)
draft_dict = dict(zip(ports_tides['port_id'], ports_tides['effective_max_draft']))

# 2. Proses Prediksi & Penugasan Kapal untuk Top 3 Rute Baru
recommendations = []
active_fleet = fleet[fleet['status'] == 'active'].copy()

# Estimasi global Baseline OPEX per Liter dan per Hari Crew dari rute R01
# (Dari eksplorasi kita tahu fuel_price ~ Rp 6,700 per liter)
FUEL_PRICE_IDR = 6703.23

for idx, route in top_3_routes.iterrows():
    # Prediksi Demand & Revenue
    X_new = pd.DataFrame({'avg_daily_mobility': [route['avg_daily_mobility']], 'distance_nm': [route['distance_nm']]})
    pred_weekly_demand = demand_model.predict(X_new)[0]
    # Pastikan demand tidak negatif
    pred_weekly_demand = max(50, pred_weekly_demand) 
    
    est_ticket_price = route['distance_nm'] * avg_price_per_nm
    est_monthly_revenue = pred_weekly_demand * 4.33 * est_ticket_price
    
    # Hitung batas draft
    origin_draft = draft_dict.get(route['origin_port_id'], 5.0)
    dest_draft = draft_dict.get(route['dest_port_id'], 5.0)
    route_max_draft = min(origin_draft, dest_draft)
    
    # Cari kapal yang memenuhi batas draft
    valid_ships = active_fleet[active_fleet['draft_m'] <= route_max_draft]
    
    best_ship = None
    best_roi = -float('inf')
    best_opex = 0
    best_freq = 0
    
    for _, ship in valid_ships.iterrows():
        # Hitung frekuensi yang dibutuhkan untuk memenuhi demand
        req_freq_per_week = max(1, int(np.ceil(pred_weekly_demand / ship['pax_capacity'])))
        
        # Hitung durasi dan OPEX
        hours_per_trip = route['distance_nm'] / ship['service_speed_knots']
        fuel_liters_per_trip = hours_per_trip * ship['fuel_consumption_lph'] * 2 # Round trip
        fuel_cost_monthly = fuel_liters_per_trip * FUEL_PRICE_IDR * req_freq_per_week * 4.33
        
        # Crew cost (alokasi proporsional berdasarkan jam jalan dari max 134 jam)
        hours_weekly = req_freq_per_week * hours_per_trip * 2
        crew_cost_monthly = (ship['daily_crew_cost_idr'] * 30) * (hours_weekly / 168.0)
        
        # Asumsi flat port fees & maintenance per trip (misal Rp 20 juta per round trip)
        port_maint_monthly = 20_000_000 * req_freq_per_week * 4.33
        
        est_monthly_opex = fuel_cost_monthly + crew_cost_monthly + port_maint_monthly
        roi = (est_monthly_revenue - est_monthly_opex) / est_monthly_opex * 100
        
        if roi > best_roi:
            best_roi = roi
            best_ship = ship['ship_id']
            best_opex = est_monthly_opex
            best_freq = req_freq_per_week
            
    if best_ship is not None:
        recommendations.append({
            'new_route_id': route['new_route_id'],
            'origin_port_id': route['origin_port_id'],
            'dest_port_id': route['dest_port_id'],
            'distance_nm': round(route['distance_nm'], 2),
            'forecast_weekly_demand': int(pred_weekly_demand),
            'est_ticket_price_idr': round(est_ticket_price, 2),
            'recommended_ship_id': best_ship,
            'recommended_frequency': best_freq,
            'est_monthly_revenue_idr': round(est_monthly_revenue, 2),
            'est_monthly_opex_idr': round(best_opex, 2),
            'projected_roi_percent': round(best_roi, 2)
        })

rec_df = pd.DataFrame(recommendations)
display(rec_df)

# Simpan ke CSV
output_path = os.path.join(output_dir, "new_route_recommendations.csv")
rec_df.to_csv(output_path, index=False)
print(f"Rekomendasi rute baru berhasil disimpan ke: {output_path}")

,new_route_id,origin_port_id,dest_port_id,distance_nm,forecast_weekly_demand,est_ticket_price_idr,recommended_ship_id,recommended_frequency,est_monthly_revenue_idr,est_monthly_opex_idr,projected_roi_percent
0,N01,PRK,MRK,67.74,24600,189218.06,KM-05,20,2.015522e+10,3.960760e+09,408.87
1,N02,PRK,BKH,87.69,21953,244942.41,KM-05,18,2.328341e+10,4.155413e+09,460.32
2,N03,PRK,TGL,181.27,14044,506358.92,KM-32,17,3.079406e+10,4.474264e+09,588.25


Rekomendasi rute baru berhasil disimpan ke: output/new_route_recommendations.csv


## 4. Kesimpulan & Rekomendasi Eksekutif

Berdasarkan perpaduan Model Spasial Mobilitas dan Regresi Permintaan, berikut adalah **3 Rute Baru Terbaik** yang direkomendasikan untuk dieksekusi oleh Bvarta Bahari:

1. **Rute N01 (Merak - Tanjung Priok)**
   Memanfaatkan tingginya mobilitas perbatasan logistik Jakarta-Banten. Jarak pelayarannya yang pendek memungkinkan operasi frekuensi tinggi dengan margin yang sangat sehat.
2. **Rute N02 (Bakauheni - Tanjung Priok)**
   Jalur *bypass* strategis untuk menghindari kemacetan pelabuhan Merak-Bakauheni. Data mobilitas menunjukkan pergerakan penumpang Sumatera-Jakarta yang sangat dominan.
3. **Rute N03 (Tanjung Priok - Tegal)**
   Rute pesisir utara Jawa. Meskipun jaraknya lebih jauh, minimnya jalur tol laut di segmen ini membuka peluang monopoli trayek penumpang antar kota dengan potensi ROI yang menjanjikan.

> **Tindakan Lanjut**:
> Direkomendasikan untuk melakukan pelayaran percobaan (*maiden voyage*) pada Rute N01 menggunakan kapal terpilih, sebelum meluncurkan armada untuk rute N02 dan N03. Analisis di atas mengasumsikan ketersediaan kapal dari armada aktif; jika kapal tersebut bertabrakan dengan jadwal rute optimasi sebelumnya, direksi disarankan melakukan *leasing* kapal dengan spesifikasi OPEX/draft yang serupa.